In [1]:
import os
import pandas as pd
from glob import glob
import numpy as np

os.chdir('/store/carroll/sbgplants/')

In [2]:
# file paths
raw = 'data/raw'

doi_loc = os.path.join(raw, '10.15485.1618130') # Locations, metadata, and species cover from field sampling survey associated with NEON AOP survey, East River, CO 2018
doi_lma = os.path.join(raw, '10.15485.1618132') # Leaf mass per area and leaf water content measurements from field survey in association with NEON AOP survey, East River, CO 2018
doi_chem = os.path.join(raw, '10.15485.1631278') # Site-level Foliar C, N, delta13C data from samples collected during field survey associated with NEON AOP survey, East River, CO 2018

out_folder = 'data/out_csv'

table = 'leaf_traits'

In [41]:
# load relevant plot/sample/spp data

# get species_or_type from spp list
species_list = pd.read_csv(os.path.join(doi_loc, 'species_list.csv'))
species_list['species_or_type'] = species_list['Genus'] + ' ' + species_list['Species']
species_list.loc[species_list['species_or_type'].isna(), 'species_or_type'] = species_list.loc[species_list['Genus'].isna(), 'CoverCode']
species_list = species_list[['CoverCode', 'species_or_type']]

# prep sample_list to get sample_id
sample = pd.read_csv(os.path.join(out_folder, 'sample.csv'))[['sample_name','species_or_type']]
sample['plot_name'] = [x.split('_')[0] for x in sample['sample_name']]
# sample_list = pd.read_csv(os.path.join(out_folder, 'sample_list.csv'))[['plot_name', 'species_or_type', 'sample_id']]

# prep sample_site to get sample_id for meadows
sample_site = pd.read_csv(os.path.join(doi_loc, 'sample_site.csv'))[['SamplingArea', 'SampleSiteCode']]

# prep sampling_area to get sample_id
sampling_area = pd.read_csv(os.path.join(doi_loc, 'sampling_area.csv'))

In [36]:
# load relevant trait data - process each of the 3 trait datasets separately and then concatenate

# start with the easiest set - trees. One sample per entry in sample_list
lma_site = pd.read_csv(os.path.join(doi_lma, 'lma_site_samples.csv'))
# fix typo
lma_site.loc[lma_site.Species=='engelmann', 'Species'] = 'Engelmann'
# join species_or_type
lma_site = pd.merge(lma_site, species_list, left_on='Species', right_on='CoverCode', how='left', suffixes=('',''))

# # assign sample_id
lma_site = pd.merge(lma_site, sample, left_on=['SampleSiteCode','species_or_type'], right_on=['plot_name','species_or_type'], how='left', suffixes=('',''))

# # short to long format
lma_site = lma_site[['Wet_Weight_g', 'Dry_Weight_g', 'LMA_gm2', 'LWC_%', 'sample_name']]
lma_site = pd.melt(lma_site, id_vars=['sample_name'], value_vars=['Wet_Weight_g', 'Dry_Weight_g', 'LMA_gm2', 'LWC_%'], var_name='trait', value_name='value')

# prepare & populate out table
out_table = pd.DataFrame(index=range(len(lma_site)))

out_table['leaf_trait_id'] = range(len(out_table))
out_table['sample_name'] = lma_site['sample_name']
out_table['trait'] = lma_site['trait']
out_table['value'] = lma_site['value']
out_table['method'] = 'Weight based'

# map trait names
traits = {
    'Wet_Weight_g': 'wet weight',
    'Dry_Weight_g': 'dry weight',
    'LMA_gm2': 'LMA',
    'LWC_%': 'LWC'
}
out_table['trait'] = out_table['trait'].map(traits)

# map sample handling
handling = {
    'wet weight': 'Fresh',
    'dry weight': 'Oven dried',
    'LMA': 'Oven dried', # ?
    'LWC': 'Oven dried' # ?
}
out_table['handling'] = out_table['trait'].map(handling)

# map units
units = {
    'wet weight': 'g',
    'dry weight': 'g',
    'LMA': 'grams dry mass per g m2',
    'LWC': 'percentage'
}
out_table['units'] = out_table['trait'].map(units)
out_table['notes'] = None
out_table['error'] = None
out_table['error_type'] = None

# filter na rows
out_table = out_table[out_table.value.isna()==False]

out_table_lmasite = out_table.copy()

out_table_lmasite

,leaf_trait_id,sample_name,trait,value,method,handling,units,notes,error,error_type
0,0,020-ER18_Piceaengelmannii,wet weight,2.08,Weight based,Fresh,g,None,None,None
1,1,021-ER18_Salixwolfii,wet weight,1.34,Weight based,Fresh,g,None,None,None
2,2,022-ER18_Salixboothii,wet weight,1.65,Weight based,Fresh,g,None,None,None
3,3,023-ER18_Salixwolfii,wet weight,1.62,Weight based,Fresh,g,None,None,None
4,4,024-ER18_Salixboothii,wet weight,1.70,Weight based,Fresh,g,None,None,None
...,...,...,...,...,...,...,...,...,...,...
1038,1038,434-ER18_Piceaengelmannii,LWC,45.60,Weight based,Oven dried,percentage,None,None,None
1039,1039,435-ER18_Abieslasiocarpa,LWC,46.20,Weight based,Oven dried,percentage,None,None,None
1040,1040,436-ER18_Populustremuloides,LWC,56.00,Weight based,Oven dried,percentage,None,None,None
1041,1041,437-ER18_Piceaengelmannii,LWC,45.10,Weight based,Oven dried,percentage,None,None,None


In [67]:
# # repeat for meadow traits
# # this is the tricky one - rows in leaf_properties will be duplicated for multiple sample_ids?
# # this is weird and maybe pseudo-replication?
# # hold off on this until talk to Dana

# lma_meadow = pd.read_csv(os.path.join(doi_lma, 'lma_meadow_area_samples.csv'))

# # join species_or_type to lma_meadow
# lma_meadow = pd.merge(lma_meadow, species_list, left_on='SpeciesCode', right_on='CoverCode', how='left', suffixes=('',''))

# # join SamplingArea to sample_list
# sample_list_ = pd.merge(sample_list, sample_site, left_on=['plot_name'], right_on=['SampleSiteCode'], how='left', suffixes=('',''))

# # get all plots per SampleArea/species_or_type
# sampleid_key = (
#     sample_list_
#     .groupby(['SamplingArea', 'species_or_type'])['sample_id']
#     .agg(list)
#     .reset_index(name='sample_id')
# )
# # assign sample ids
# lma_meadow = lma_meadow.merge(
#     sampleid_key,
#     left_on=['SampleArea', 'species_or_type'],
#     right_on=['SamplingArea', 'species_or_type'],
#     how='left'
# )

# # short to long format
# lma_meadow = lma_meadow[['Wet Weight (g)', 'Dry Weight (g)', 'LMA (g/m2)', 'LWC (%)', 'sample_id']]
# lma_meadow = pd.melt(lma_meadow, id_vars=['sample_id'], value_vars=['Wet Weight (g)', 'Dry Weight (g)', 'LMA (g/m2)', 'LWC (%)'], var_name='trait', value_name='value')

# # prepare & populate out table
# out_table = pd.DataFrame(columns=schema['column_name'].unique())

# out_table['trait'] = lma_meadow['trait']
# out_table['value'] = lma_meadow['value']
# out_table['sample_id'] = lma_meadow['sample_id']

# out_table['method'] = 'Destructive' # double-check

# # map trait names
# traits = {
#     'Wet Weight (g)': 'wet weight',
#     'Dry Weight (g)': 'dry weight',
#     'LMA (g/m2)': 'LMA',
#     'LWC (%)': 'LWC'
# }
# out_table['trait'] = out_table['trait'].map(traits)

# # map sample handling
# handling = {
#     'wet weight': 'Fresh',
#     'dry weight': 'Oven Dried',
#     'LMA': 'Oven Dried', # ?
#     'LWC': pd.NA # ?
# }
# out_table['handling'] = out_table['trait'].map(handling)

# # map units
# units = {
#     'wet weight': 'g',
#     'dry weight': 'g',
#     'LMA': 'grams dry mass per g m2',
#     'LWC': 'percentage'
# }
# out_table['units'] = out_table['trait'].map(units)

# # filter na rows
# out_table = out_table[out_table.value.isna()==False]

# out_table_lmameadow = out_table.copy()

In [44]:
# # repeat for foliar chemistry
# chem = pd.read_csv(os.path.join(doi_chem, 'CN_Results_Foliar.csv'))

# # get all sample_ids per plot

# # join SamplingArea to sample_list
# sample_list_ = pd.merge(sample, sample_site, left_on=['plot_name'], right_on=['SampleSiteCode'], how='left', suffixes=('',''))

# # sampleid_key = (
# #     sample_list_
# #       .groupby('plot_name', sort=False)['sample_id']
# #       .agg(list)              # unique, order-preserving
# # )
# # # assign sample ids
# # chem['sample_id'] = chem['SampleSiteCode'].map(sampleid_key)
# sampleid_key = (
#     sample_list_
#       .groupby('plot_name', sort=False)['sample_name']
#       .agg(list)              # unique, order-preserving
# )
# # assign sample ids
# chem['sample_name'] = chem['SampleSiteCode'].map(sampleid_key)

# # # short to long format
# # chem = chem[['N_weight_percent', 'sample_id']] # no C_weight_percent or d13C?
# # chem = pd.melt(chem, id_vars=['sample_id'], value_vars=['N_weight_percent'], var_name='trait', value_name='value')

# # # prepare & populate out table
# # out_table = pd.DataFrame(columns=schema['column_name'].unique())

# # out_table['trait'] = chem['trait']
# # out_table['value'] = chem['value']
# # out_table['sample_id'] = chem['sample_id']

# # out_table['method'] = 'Destructive' # double-check

# # # map trait names
# # traits = {
# #     'N_weight_percent': 'Nitrogen'
# # }
# # out_table['trait'] = out_table['trait'].map(traits)

# # # map sample handling
# # handling = {
# #     'Nitrogen': 'Flash frozen'
# # }
# # out_table['handling'] = out_table['trait'].map(handling)

# # # map units
# # units = {
# #     'Nitrogen': 'concentration in percent dry mass' # vs just percent?
# # }
# # out_table['units'] = out_table['trait'].map(units)

# # # # filter na rows
# # # out_table = out_table[out_table.value.isna()==False]

# # out_table_chem = out_table.copy()

# # out_table_chem

# chem

In [45]:
# # join the three datasets

# out_table = pd.concat([out_table_lmasite, out_table_lmameadow, out_table_chem])
# out_table

In [47]:
# export table
fp_out = os.path.join(out_folder, f'{table}.csv')
out_table.to_csv(fp_out, index=False)

In [46]:
out_table

,leaf_trait_id,sample_name,trait,value,method,handling,units,notes,error,error_type
0,0,020-ER18_Piceaengelmannii,wet weight,2.08,Weight based,Fresh,g,None,None,None
1,1,021-ER18_Salixwolfii,wet weight,1.34,Weight based,Fresh,g,None,None,None
2,2,022-ER18_Salixboothii,wet weight,1.65,Weight based,Fresh,g,None,None,None
3,3,023-ER18_Salixwolfii,wet weight,1.62,Weight based,Fresh,g,None,None,None
4,4,024-ER18_Salixboothii,wet weight,1.70,Weight based,Fresh,g,None,None,None
...,...,...,...,...,...,...,...,...,...,...
1038,1038,434-ER18_Piceaengelmannii,LWC,45.60,Weight based,Oven dried,percentage,None,None,None
1039,1039,435-ER18_Abieslasiocarpa,LWC,46.20,Weight based,Oven dried,percentage,None,None,None
1040,1040,436-ER18_Populustremuloides,LWC,56.00,Weight based,Oven dried,percentage,None,None,None
1041,1041,437-ER18_Piceaengelmannii,LWC,45.10,Weight based,Oven dried,percentage,None,None,None
